# 🔬 DeepTrace v2 — EfficientNet-B4 Training Pipeline

This notebook trains an **EfficientNet-B4** model for deepfake detection using the
**[ScaleDF](https://huggingface.co/datasets/WenhaoWang/ScaleDF)** dataset — the largest
deepfake detection dataset available (5.8M real images from 51 sources, 8.8M fake images
from 102 generation methods including GANs, diffusion models, and face swaps).

> **Why ScaleDF over xhlulu/140k-real-and-fake-faces?** The 140k dataset only contains
> FFHQ photos vs. old StyleGAN fakes. A model trained on it learns GAN-specific frequency
> artifacts instead of generalizable forensic cues. ScaleDF covers 102 manipulation methods,
> forcing the model to learn subtle, method-agnostic indicators that survive social media
> compression — which is exactly what DeepTrace needs in production.

The data **streams directly from HuggingFace** via the `webdataset` library — no manual
download or disk space needed for the training set.

> **Reviewed & hardened.** This notebook incorporates fixes for: an HF `Trainer`
> column-pruning crash, Albumentations 2.x API compatibility, EfficientNet-B4
> normalization mismatch, and label-mapping assertions that prevent silent production bugs.

### ⚡ Prerequisites
- Set your Colab Runtime to **T4 GPU** or **A100 GPU** (`Runtime > Change runtime type`)
- No Kaggle credentials needed — ScaleDF is hosted on HuggingFace Hub

In [ ]:
# Step 1: Install Required Dependencies
# albumentations pinned to 2.0.8 — last MIT-licensed release before AGPL fork.
# opencv-python-headless avoids libGL errors on Colab's GUI-less runtime.
# webdataset streams ScaleDF's tar shards without downloading the full 2TB+ dataset.
!pip install -q transformers datasets accelerate torchvision evaluate scikit-learn \
    "albumentations==2.0.8" opencv-python-headless webdataset huggingface_hub

## 1. Environment & Reproducibility

In [ ]:
import os
import io
import random
import numpy as np
import torch
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    torch.cuda.manual_seed_all(SEED)
else:
    print("WARNING: No GPU detected...")

os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
print("HuggingFace token successfully loaded!")

## 2. Load EfficientNet-B4 Processor

We load the processor **before** building the augmentation pipeline. It carries the exact
normalization statistics (`image_mean` / `image_std`) and input resolution baked into the
`google/efficientnet-b4` checkpoint.

> ⚠️ **Silent production bug prevention:** `google/efficientnet-b4`'s `image_std` is
> `[0.4785, 0.4733, 0.4743]`, **not** the generic ImageNet `[0.229, 0.224, 0.225]` most
> tutorials hardcode. Since `processor.save_pretrained()` is what your FastAPI backend loads,
> training with one normalization and shipping a config with a different one silently skews
> every prediction. We derive everything from the processor object below.

In [ ]:
from transformers import AutoImageProcessor

MODEL_ID = "google/efficientnet-b4"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

IMAGE_SIZE = processor.size["height"]  # 380 for B4 — derived, not hardcoded
NORM_MEAN = processor.image_mean
NORM_STD = processor.image_std

print(f"Resolution:     {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Normalize mean: {NORM_MEAN}")
print(f"Normalize std:  {NORM_STD}")

## 3. Label Mapping

The backend's `analyzer.py` looks up `self.model.config.label2id.get("Fake", 1)` —
exact capitalized strings matter. We define this once and thread it through every step
so a typo fails here instead of silently flipping predictions in production.

In [ ]:
id2label = {0: "Real", 1: "Fake"}
label2id = {"Real": 0, "Fake": 1}

assert set(id2label.values()) == {"Real", "Fake"}
assert label2id["Fake"] != label2id["Real"]
print(f"Label mapping: {id2label}")

## 4. Forensic Augmentations

To survive WhatsApp and Instagram compression, we simulate social media degradation
during training using `albumentations`.

> **Albumentations API note:** All transforms below use the current 2.x argument names
> (`quality_range`, `std_range`, `size=`, `brightness_range`, etc.). The old 1.x names
> (`quality_lower`, `var_limit`, `height=`, `brightness=`) raise `TypeError` at compose time.
> Pinned against `albumentations==2.0.8` (installed above).

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    
    # 1. FIXED: Replaced ShiftScaleRotate with Affine (and correct parameters)
    A.Affine(
        translate_percent=(-0.05, 0.05),
        scale=(0.95, 1.05),
        rotate=(-15, 15),
        p=0.5,
    ),

    # 2. FIXED: ColorJitter parameters (removed the '_range' suffix)
    A.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2),
                  saturation=(0.8, 1.2), hue=(-0.1, 0.1), p=0.5),

    # ⚡ CRITICAL: Social Media Compression Simulation ⚡
    A.ImageCompression(quality_range=(30, 90), p=0.6),
    
    # 3. FIXED: GaussianBlur uses 'blur_limit'
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(std_range=(0.03, 0.12), p=0.3),

    # Partial occlusion (hands, hair, masks) — stops the model leaning on one region
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(0.05, 0.15),
                     hole_width_range=(0.05, 0.15), p=0.15),

    # Spatial
    A.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.8, 1.0)),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

val_augmentations = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

print("Augmentation pipeline initialized.")

## 5. ScaleDF Dataset — Streaming from HuggingFace

ScaleDF is distributed as WebDataset `.tar` shards on HuggingFace Hub:
- **Real face** shards have filenames starting with `000000` (e.g. `000000AFAD.tar`)
- **Fake face** shards use other names (e.g. `AMatrix_faces.tar`, `StableDiffusion_faces.tar`)

We stream the shards directly via HTTP — no download to disk needed. A balanced 50/50
real/fake sampler ensures the model sees equal amounts of each class regardless of the
underlying class imbalance (5.8M real vs. 8.8M fake).

> **Streaming means no epochs.** Training runs for a fixed number of `max_steps` instead.
> Each step pulls a fresh batch from the shard stream. Over 15,000 steps with effective
> batch size 32, the model sees ~480,000 images from diverse methods — already 3.4× more
> than the entire 140k dataset, and far more diverse.

In [ ]:
import webdataset as wds
from huggingface_hub import HfApi

api = HfApi()

# ── Discover training shards ──
print("Listing ScaleDF training shards from HuggingFace Hub...")
train_entries = list(api.list_repo_tree(
    "WenhaoWang/ScaleDF", repo_type="dataset", path_in_repo="ScaleDF/train"
))
train_tars = sorted([e.rfilename for e in train_entries if e.rfilename.endswith(".tar")])

# Separate real (000000*) from fake
real_shard_names = [t for t in train_tars if t.split("/")[-1].startswith("000000")]
fake_shard_names = [t for t in train_tars if not t.split("/")[-1].startswith("000000")]

print(f"Real source domains:     {len(real_shard_names)}")
print(f"Fake generation methods: {len(fake_shard_names)}")
print(f"Total training shards:   {len(train_tars)}")

# ── Discover validation shards ──
print("\nListing ScaleDF validation shards...")
val_entries = list(api.list_repo_tree(
    "WenhaoWang/ScaleDF", repo_type="dataset", path_in_repo="ScaleDF/val"
))
val_tars = sorted([e.rfilename for e in val_entries if e.rfilename.endswith(".tar")])

val_real_names = [t for t in val_tars if t.split("/")[-1].startswith("000000")]
val_fake_names = [t for t in val_tars if not t.split("/")[-1].startswith("000000")]

print(f"Val real shards:  {len(val_real_names)}")
print(f"Val fake shards:  {len(val_fake_names)}")

# ── Build full URLs ──
BASE_URL = "https://huggingface.co/datasets/WenhaoWang/ScaleDF/resolve/main/"

# If HF_TOKEN is set, append it so webdataset gets authenticated downloads
import os
_hf_token = os.environ.get("HF_TOKEN", "")
_url_suffix = f"?token={_hf_token}" if _hf_token else ""
real_train_urls = [BASE_URL + name + _url_suffix for name in real_shard_names]
fake_train_urls = [BASE_URL + name + _url_suffix for name in fake_shard_names]
real_val_urls   = [BASE_URL + name + _url_suffix for name in val_real_names]
fake_val_urls   = [BASE_URL + name + _url_suffix for name in val_fake_names]

print(f"\nSample real shards: {[s.split('/')[-1] for s in real_shard_names[:3]]}")
print(f"Sample fake shards: {[s.split('/')[-1] for s in fake_shard_names[:3]]}")

In [ ]:
from PIL import Image
import io
import numpy as np
import torch
import random
import webdataset as wds

# ── Image extraction helper ──
def extract_image(sample):
    for key in ("jpg", "jpeg", "png", "webp", "bmp", "tiff", "ppm"):
        if key in sample and isinstance(sample[key], Image.Image):
            return sample[key]
    for key, val in sample.items():
        if key.startswith("__"):
            continue
        if isinstance(val, Image.Image):
            return val
        if isinstance(val, bytes) and len(val) > 100:
            try:
                return Image.open(io.BytesIO(val))
            except Exception:
                continue
    return None

def _validate_image(img):
    if img is None:
        return None
    try:
        arr = np.array(img.convert("RGB"))
        if arr.ndim != 3 or arr.shape[2] != 3:
            return None
        if min(arr.shape[:2]) < 20:
            return None
        return arr
    except Exception:
        return None

# Streaming Training Dataset
class ScaleDFTrainDataset(torch.utils.data.IterableDataset):
    def __init__(self, real_urls, fake_urls, augmentation, label2id):
        super().__init__()
        self.real_urls = list(real_urls)
        self.fake_urls = list(fake_urls)
        self.augmentation = augmentation
        self.label2id = label2id

    def _make_pipeline(self, urls):
        return (
            wds.WebDataset(
                urls,
                resampled=True,
                shardshuffle=False,
                handler=wds.warn_and_continue,
            )
            .shuffle(1000, handler=wds.warn_and_continue)
            .decode("pil", handler=wds.warn_and_continue)
        )

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        real_urls = self.real_urls
        fake_urls = self.fake_urls
        if worker_info is not None:
            n = worker_info.num_workers
            w = worker_info.id
            real_urls = self.real_urls[w::n] or self.real_urls
            fake_urls = self.fake_urls[w::n] or self.fake_urls

        real_iter = iter(self._make_pipeline(real_urls))
        fake_iter = iter(self._make_pipeline(fake_urls))

        while True:
            if random.random() < 0.5:
                sample = next(real_iter)
                label = self.label2id["Real"]
            else:
                sample = next(fake_iter)
                label = self.label2id["Fake"]

            img = extract_image(sample)
            arr = _validate_image(img)
            if arr is None:
                continue

            try:
                pixel_values = self.augmentation(image=arr)["image"]
                yield {"pixel_values": pixel_values, "label": label}
            except Exception:
                continue

# Fixed-size Validation Dataset
class ScaleDFValDataset(torch.utils.data.Dataset):
    def __init__(self, real_urls, fake_urls, augmentation, label2id, max_per_class=1000):
        super().__init__()
        self.augmentation = augmentation
        self.items = []

        for class_name, urls, target in [("real", real_urls, "Real"), ("fake", fake_urls, "Fake")]:
            print(f"  Collecting up to {max_per_class} {class_name} validation samples...")
            count = 0
            if not urls:
                print(f"  WARNING: no {class_name} val shards found -- skipping.")
                continue
            
            # FIXED: Explicitly set shardshuffle=False to silence warning
            pipe = wds.WebDataset(urls, shardshuffle=False, handler=wds.warn_and_continue)\
                .decode("pil", handler=wds.warn_and_continue)
            
            for sample in pipe:
                if count >= max_per_class:
                    break
                img = extract_image(sample)
                arr = _validate_image(img)
                if arr is not None:
                    self.items.append((img.convert("RGB"), label2id[target]))
                    count += 1
            print(f"  Collected {count} {class_name} samples.")

        random.shuffle(self.items)
        real_n = sum(1 for _, l in self.items if l == label2id["Real"])
        fake_n = sum(1 for _, l in self.items if l == label2id["Fake"])
        print(f"  Validation set: {real_n} real + {fake_n} fake = {len(self.items)} total")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img, label = self.items[idx]
        arr = np.array(img)
        pixel_values = self.augmentation(image=arr)["image"]
        return {"pixel_values": pixel_values, "label": label}

In [ ]:
# ── Instantiate datasets ──
print("Creating training dataset (streaming)...")
train_ds = ScaleDFTrainDataset(real_train_urls, fake_train_urls, train_augmentations, label2id)

print("Creating validation dataset (downloading to memory -- may take a few minutes)...")
val_ds = ScaleDFValDataset(real_val_urls, fake_val_urls, val_augmentations, label2id, max_per_class=1000)

assert len(val_ds) > 0, (
    "Validation dataset is empty. Check your network connection to HuggingFace Hub. "
    "If rate-limited, set your HF token: huggingface-cli login"
)

print(f"\nDatasets ready! Val size: {len(val_ds)}")

## 6. Sanity Check — Verify One Sample

Quick check that the streaming pipeline is working before committing to a multi-hour
training run. You should see a tensor of shape `(3, 380, 380)` and a valid label.

In [ ]:
# Verify one sample from the training stream
print("Pulling one training sample from the shard stream...")
sample = next(iter(train_ds))
print(f"  pixel_values shape: {sample['pixel_values'].shape}")
print(f"  pixel_values dtype: {sample['pixel_values'].dtype}")
print(f"  label: {sample['label']} ({id2label[sample['label']]})")

assert sample["pixel_values"].shape == (3, IMAGE_SIZE, IMAGE_SIZE), \
    f"Expected (3, {IMAGE_SIZE}, {IMAGE_SIZE}), got {sample['pixel_values'].shape}"

# Verify one sample from the val dataset
val_sample = val_ds[0]
print(f"\n  val pixel_values shape: {val_sample['pixel_values'].shape}")
print(f"  val label: {val_sample['label']} ({id2label[val_sample['label']]})")

print("\nPipeline is working correctly!")

## 6.5 Visualize Training Samples

Plot a few augmented training samples to verify the pipeline (compression, blur, normalization).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax in axes.flat:
    s = next(iter(train_ds))
    img = s["pixel_values"].permute(1, 2, 0).numpy()
    img = img * np.array(NORM_STD) + np.array(NORM_MEAN) # denormalize
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(id2label[s["label"]])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. EfficientNet-B4 Model Setup

We use HuggingFace's `AutoModelForImageClassification` to ensure 100% compatibility
with `analyzer.py`. The final classification head is replaced with a 2-class head
(`Real` / `Fake`) using the label mapping defined above.

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print("EfficientNet-B4 loaded successfully!")
print(f"model.config.label2id = {model.config.label2id}")

## 8. Training Configuration

Key design decisions:
- **`max_steps` instead of `num_train_epochs`**: streaming datasets have no fixed length
- **`learning_rate=3e-5`**: full-model fine-tuning (nothing frozen); higher rates risk
  destroying the pretrained ImageNet features
- **`remove_unused_columns=False`**: Trainer would otherwise drop columns the model's
  `forward()` doesn't accept *before* our transform builds `pixel_values`
- **`EarlyStoppingCallback`**: halts training if val F1 stalls for 3 evaluations
- **`fp16` / `bf16` auto-switch**: T4 uses fp16, A100 uses bf16 (more numerically stable)

In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="weighted")["recall"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([ex["pixel_values"] for ex in examples])
    labels = torch.tensor([ex["label"] for ex in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

OUTPUT_DIR = "./deeptrace-efficientnet"

# T4 has no bf16 tensor cores; A100+ supports bf16 (more numerically stable).
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Streaming: use max_steps instead of num_train_epochs.
    # 15,000 steps x 32 effective batch = ~480,000 images (~4-8 hours on T4).
    # Increase to 30,000 if you have time.
    max_steps=15000,

    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,

    learning_rate=3e-5,
    weight_decay=1e-5,
    lr_scheduler_type="cosine",

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,   # Effective batch size = 32
    per_device_eval_batch_size=32,

    # Flip to True if you hit CUDA OOM on the T4
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    warmup_steps=750,
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,

    fp16=use_fp16,
    bf16=use_bf16,
    label_smoothing_factor=0.1,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Training for up to {training_args.max_steps:,} steps")
print(f"Eval every {training_args.eval_steps:,} steps")
print(f"Early stopping patience: 3 evaluations ({3 * training_args.eval_steps:,} steps)")

In [ ]:
# 🚀 Run Training
# To resume after a Colab disconnect, change the line below to:
#   trainer.train(resume_from_checkpoint=True)
print("Starting EfficientNet-B4 fine-tuning on ScaleDF...")
trainer.train()

## 9. Evaluate on Validation Set

Aggregate metrics plus a confusion matrix and per-class report. For a forensic tool,
the two error types are not symmetric: a **false negative** (a deepfake classified as real)
is the costlier mistake, so we report that rate explicitly.

In [ ]:
# 📊 Evaluate
print("Evaluating on validation set...")
metrics = trainer.evaluate()
for k, v in sorted(metrics.items()):
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Per-class breakdown
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(val_ds)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids
target_names = [id2label[i] for i in range(len(id2label))]

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows = true, cols = predicted):")
header = f"{'':>8}" + "".join(f"{n:>8}" for n in target_names)
print(header)
for i, row in enumerate(cm):
    print(f"{target_names[i]:>8}" + "".join(f"{v:>8}" for v in row))

fake_idx = label2id["Fake"]
real_idx = label2id["Real"]
false_negatives = cm[fake_idx][real_idx]
print(f"\nFalse negatives (Fake classified as Real): {false_negatives} / {cm[fake_idx].sum()}")
print(f"False positive rate (Real classified as Fake): {cm[real_idx][fake_idx]} / {cm[real_idx].sum()}")

## 9.5 Evaluate on Wild_Deepfake (Real-world Holdout)

Evaluate the model on a completely unseen distribution of internet deepfakes to test generalizability.

In [ ]:
# Download and evaluate on Wild_Deepfake
import tarfile
import urllib.request
import os

print("Downloading Wild_Deepfake benchmark...")
wild_url = "https://huggingface.co/datasets/WenhaoWang/ScaleDF/resolve/main/Established_benchmarks/Wild_Deepfake.tar"
_hf_token = os.environ.get("HF_TOKEN", "")
if _hf_token:
    wild_url += f"?token={_hf_token}"

print("Evaluating on Wild_Deepfake stream...")
wild_ds = ScaleDFValDataset([], [wild_url], val_augmentations, label2id, max_per_class=2000)

if len(wild_ds) > 0:
    wild_predictions = trainer.predict(wild_ds)
    wild_y_pred = np.argmax(wild_predictions.predictions, axis=1)
    wild_y_true = wild_predictions.label_ids
    
    print("\nWild_Deepfake Classification report:")
    print(classification_report(wild_y_true, wild_y_pred, target_names=target_names, digits=4))
    
    wild_cm = confusion_matrix(wild_y_true, wild_y_pred)
    print("Wild_Deepfake Confusion matrix (rows = true, cols = predicted):")
    print(header)
    for i, row in enumerate(wild_cm):
        print(f"{target_names[i]:>8}" + "".join(f"{v:>8}" for v in row))
else:
    print("Failed to load Wild_Deepfake.")


## 10. Save, Verify & Export

Before zipping, we reload the saved checkpoint and confirm `label2id` round-trips
to exactly `{"Fake": ..., "Real": ...}` and the normalization matches the processor.
This catches a broken export here rather than after deploying to the backend.

In [ ]:
# 💾 Save and Export Model
import shutil
from google.colab import files

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Round-trip check: reload exactly like the FastAPI backend would
_check_model = AutoModelForImageClassification.from_pretrained(OUTPUT_DIR)
_check_processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR)
assert _check_model.config.label2id.get("Fake") is not None, "Saved config missing Fake label!"
assert _check_model.config.label2id.get("Real") is not None, "Saved config missing Real label!"
assert list(_check_processor.image_mean) == list(NORM_MEAN), \
    f"Norm mean mismatch: saved={_check_processor.image_mean}, expected={NORM_MEAN}"
assert list(_check_processor.image_std) == list(NORM_STD), \
    f"Norm std mismatch: saved={_check_processor.image_std}, expected={NORM_STD}"
print(f"Saved label2id:     {_check_model.config.label2id}")
print(f"Saved norm mean:    {_check_processor.image_mean}")
print(f"Saved norm std:     {_check_processor.image_std}")
del _check_model, _check_processor

zip_filename = "deeptrace_efficientnet.zip"
shutil.make_archive("deeptrace_efficientnet", "zip", OUTPUT_DIR)

print(f"\nCreated {zip_filename}.")
try:
    files.download(zip_filename)
    print("Downloading via browser...")
except Exception:
    print("Running on Kaggle: Download the zip from the 'Output' tab on your notebook page!")
    
print("Extract into backend/models/deeptrace-efficientnet and set DEEPTRACE_MODEL in .env!")